# 02 — Unsupervised Learning: EDA dan Evaluasi Clustering

Dataset: Mall Customers, Wholesale Customers, dan Iris.

Evaluasi clustering berbeda dari evaluasi klasifikasi. Untuk data tanpa label ground truth gunakan metrik internal (Silhouette, Davies–Bouldin, Calinski–Harabasz). Untuk Iris, species hanya digunakan sebagai evaluasi eksternal setelah clustering, bukan sebagai fitur model.


## 1. Instalasi dan import

In [ ]:
!pip -q install pandas numpy scikit-learn openpyxl


In [ ]:
import io, zipfile, urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)


## 2. Mall Customers

Sumber data: [Kaggle — Customer Segmentation Tutorial](https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python). Unggah file `Mall_Customers.csv` saat cell berikut meminta file.


In [ ]:
from google.colab import files
uploaded = files.upload()
mall_filename = next(iter(uploaded))
mall_df = pd.read_csv(mall_filename)
print("Ukuran:", mall_df.shape)
display(mall_df.head())
display(mall_df.isna().sum().to_frame("missing"))


In [ ]:
mall_features = mall_df.select_dtypes(include=np.number).copy()
mall_features = mall_features.drop(
    columns=[c for c in mall_features.columns if c.lower().replace("_", "") in {"customerid", "customer"}],
    errors="ignore"
)
print("Fitur clustering:", list(mall_features.columns))
display(mall_features.describe().T)


## 3. Wholesale Customers

Sumber resmi: [UCI Wholesale Customers](https://archive.ics.uci.edu/dataset/292/wholesale+customers). File Excel diunduh dari arsip UCI.


In [ ]:
wholesale_url = "https://archive.ics.uci.edu/static/public/292/wholesale+customers.zip"
with urllib.request.urlopen(wholesale_url) as response:
    wholesale_bytes = response.read()
with zipfile.ZipFile(io.BytesIO(wholesale_bytes)) as z:
    excel_name = next(n for n in z.namelist() if n.lower().endswith((".xlsx", ".xls")))
    with z.open(excel_name) as f:
        wholesale_df = pd.read_excel(f)
print("Ukuran:", wholesale_df.shape)
display(wholesale_df.head())
display(wholesale_df.isna().sum().to_frame("missing"))


In [ ]:
wholesale_features = wholesale_df.drop(
    columns=[c for c in ["Channel", "Region"] if c in wholesale_df.columns],
    errors="ignore"
).select_dtypes(include=np.number)
print("Fitur clustering:", list(wholesale_features.columns))
display(wholesale_features.describe().T)


## 4. Iris

Iris dimuat dari scikit-learn. Species asli tidak digunakan saat clustering, tetapi disimpan untuk evaluasi eksternal.


In [ ]:
from sklearn.datasets import load_iris
iris = load_iris(as_frame=True)
iris_X = iris.data.copy()
iris_y_reference = iris.target
print("Ukuran fitur:", iris_X.shape)
display(iris_X.head())


## 5. Evaluasi K-Means untuk beberapa nilai K

- **Silhouette**: rentang -1 sampai 1; semakin tinggi umumnya semakin baik.
- **Davies–Bouldin**: semakin rendah semakin baik.
- **Calinski–Harabasz**: semakin tinggi umumnya semakin baik.

Gunakan ketiganya sebagai bahan pertimbangan, bukan satu-satunya bukti kualitas cluster.


In [ ]:
def evaluate_kmeans(features, dataset_name, k_values=range(2, 7)):
    X = features.select_dtypes(include=np.number).copy()
    X = X.replace([np.inf, -np.inf], np.nan).dropna()
    X_scaled = StandardScaler().fit_transform(X)

    rows, fitted = [], {}
    for k in k_values:
        model = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = model.fit_predict(X_scaled)
        if len(np.unique(labels)) < 2:
            continue
        rows.append({
            "dataset": dataset_name,
            "k": k,
            "silhouette": silhouette_score(X_scaled, labels),
            "davies_bouldin": davies_bouldin_score(X_scaled, labels),
            "calinski_harabasz": calinski_harabasz_score(X_scaled, labels)
        })
        fitted[k] = (model, labels, X_scaled)

    scores = pd.DataFrame(rows)
    display(scores)
    best_k = int(scores.sort_values("silhouette", ascending=False).iloc[0]["k"])
    print("K dengan silhouette tertinggi pada rentang uji:", best_k)
    return scores, fitted, best_k

mall_scores, mall_models, mall_best_k = evaluate_kmeans(mall_features, "Mall Customers")


In [ ]:
wholesale_scores, wholesale_models, wholesale_best_k = evaluate_kmeans(
    wholesale_features, "Wholesale Customers"
)


In [ ]:
iris_scores, iris_models, iris_best_k = evaluate_kmeans(iris_X, "Iris")


## 6. Evaluasi eksternal Iris

Adjusted Rand Index (ARI) dan Normalized Mutual Information (NMI) membandingkan hasil cluster dengan species asli. Species tidak dipakai untuk membentuk cluster. Nilai yang lebih mendekati 1 menunjukkan kesesuaian lebih tinggi.


In [ ]:
iris_model, iris_cluster_labels, iris_scaled = iris_models[iris_best_k]
print("ARI:", round(adjusted_rand_score(iris_y_reference, iris_cluster_labels), 4))
print("NMI:", round(normalized_mutual_info_score(iris_y_reference, iris_cluster_labels), 4))


## 7. Referensi

1. Kaggle. *Customer Segmentation Tutorial — Mall Customers*. https://www.kaggle.com/datasets/vjchoudhary7/customer-segmentation-tutorial-in-python
2. UCI Machine Learning Repository. *Wholesale Customers*. https://archive.ics.uci.edu/dataset/292/wholesale+customers
3. Fisher, R. A. (1936). *The use of multiple measurements in taxonomic problems*. Annals of Eugenics. Iris documentation: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html
4. scikit-learn developers. *Clustering performance evaluation*. https://scikit-learn.org/stable/modules/clustering.html

Mall Customers dan Wholesale Customers tidak memiliki label target untuk evaluasi supervised. Jangan menyebut jumlah cluster sebagai jumlah label asli.
